In [ ]:
# Configurar el motor de Keras antes de importarlo
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout
from keras.regularizers import L1L2
from keras.optimizers import SGD
from keras.callbacks import EarlyStopping

# Opciones de visualización de tablas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Comprobar el acceso a la GPU
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Placa:", torch.cuda.get_device_name(0))

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view


def histograma_12_bits(fragmento, normalizar=False):
    """Recibe bytes o un vector de valores enteros entre 0 y 255.

    Devuelve 16384 conteos, o frecuencias relativas si normalizar=True.
    Un fragmento de menos de 2 bytes produce un vector de ceros.
    """
    if isinstance(fragmento, (bytes, bytearray, memoryview)):
        datos = np.frombuffer(fragmento, dtype=np.uint8)
    else:
        datos = np.asarray(fragmento)

    if datos.ndim != 1:
        raise ValueError("Se espera un solo fragmento: un vector de bytes.")
    if datos.dtype.kind not in "iuf":
        raise ValueError("Los bytes deben ser números enteros entre 0 y 255.")
    if not np.all(np.isfinite(datos)) or np.any((datos < 0) | (datos > 255)):
        raise ValueError("Los bytes deben estar entre 0 y 255, sin NaN ni infinitos.")
    if np.any(datos != np.floor(datos)):
        raise ValueError("Usá los bytes originales, antes de normalizar o escalar.")

    # Cada byte se expande a 8 bits, empezando por el más significativo.
    bits = np.unpackbits(datos.astype(np.uint8), bitorder="big")
    histograma = np.zeros(16384, dtype=np.int64)

    if bits.size >= 13:
        # Inicios en los bits 0, 4, 8, 12, ...; solo ventanas completas.
        ventanas = sliding_window_view(bits, 13)[::2]

        # Los 12 bits representan un índice entre 0 y 4095.
        pesos = 2 ** np.arange(12, -1, -1, dtype=np.uint16)
        valores = ventanas @ pesos
        histograma = np.bincount(valores, minlength=16384)

    if normalizar:
        histograma = histograma.astype(np.float32)
        total = histograma.sum()
        if total > 0:
            histograma /= total

    return histograma


if __name__ == "__main__":
    ejemplo = bytes.fromhex("AB CD EF")
    histograma = histograma_12_bits(ejemplo)
    print("Ventanas:", [format(i, "03X") for i in np.flatnonzero(histograma)])
    print("Cantidad de ventanas:", histograma.sum())
    print("Cantidad de características:", histograma.shape[0])



In [ ]:
#train = pd.read_csv(r"C:\Users\juane\OneDrive\Desktop\Redes FeedForward\clasificacion_train\dataset_8000_filas_256_bytes.csv", header=None, skipinitialspace=True)
train = pd.read_csv(r"C:\Users\juane\OneDrive\Desktop\Redes FeedForward\clasificacion_train\dataset.csv", header=None, skipinitialspace=True)

X_data = train.iloc[:, :-1].to_numpy("float32")
y_data = train.iloc[:, -1].to_numpy()               # "gif", "jpg", "png", "qoi" o "webp"

test_publico = pd.read_csv(r"C:\Users\juane\OneDrive\Desktop\Redes FeedForward\clasificacion_train\test_publico.csv", header=None)

X_test = test_publico.iloc[:, :-1].to_numpy(dtype=np.float32)
y_test_original = test_publico.iloc[:, -1].to_numpy()

X_test_entrega = pd.read_csv(r"C:\Users\juane\OneDrive\Desktop\Redes FeedForward\clasificacion_train\clasificacion_test_features.csv", header=None).to_numpy(dtype=np.float32)


In [ ]:
def miModelo(X_data, y_data, X_test, y_test, X_test_entrega):
    """X_data, y_data, X_test"""
    
    #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!LOS HISTOGRAMAS PASARON A SER DE 13 BITS Y CON VENATAS QUE SE DESPLAZAN DE A UN UNICO BIT
    # GENERA UN HISTOGRAMA POR CADA FILA
    """Se pasa a bits los bytes por fila, luego se construye un histograma de frecuencias de paquetes de 14 bits, la 'ventana' de bits se desplaza
    sobre la cadena de toda la fila de datos de a 2 bits. Cada uno de estas ventanas de 14 bits produce un nro, que se acumula en el histograma de
    frecuencias. Salida: vector en que cada posicion devuelve un conteo con la cantidad de apariciones del nro en cuestion."""

    X_hist_train = np.stack([
        histograma_12_bits(fila, normalizar=False)
        for fila in np.asarray(X_data)
    ]).astype(np.float32)

    X_hist_test = np.stack([
        histograma_12_bits(fila, normalizar=False)
        for fila in np.asarray(X_test)
    ]).astype(np.float32)

    X_hist_test_entrega = np.stack([
        histograma_12_bits(fila, normalizar=False)
        for fila in np.asarray(X_test_entrega)
    ]).astype(np.float32)
            
    # Limpiar las etiquetas
    y_data = np.char.strip(y_data.astype(str))
    y_test = np.char.strip(y_test.astype(str))

    # Crear codificador
    encoder_y = OneHotEncoder(sparse_output=False)

    # Codificar las clases
    y_train = encoder_y.fit_transform(y_data.reshape(-1, 1))
    y_test = encoder_y.transform(y_test.reshape(-1, 1))


    #ARQUITECTURA DE LA RED
    modelo = Sequential()
    
    modelo.add(Dense(32, input_dim=16384, activation="leaky_relu", kernel_regularizer=L1L2(1e-4)))
    modelo.add(Dropout(0.225))
    modelo.add(Dense(32, input_dim=32, activation="leaky_relu", kernel_regularizer=L1L2(1e-4)))
    modelo.add(Dropout(0.225))
    modelo.add(Dense(32, input_dim=32, activation="leaky_relu", kernel_regularizer=L1L2(1e-4)))
    modelo.add(Dropout(0.225))
    modelo.add(Dense(32, input_dim=32, activation="leaky_relu", kernel_regularizer=L1L2(1e-4)))
    modelo.add(Dropout(0.225))
    modelo.add(Dense(32, input_dim=32, activation="leaky_relu", kernel_regularizer=L1L2(1e-4)))
    modelo.add(Dropout(0.225))

    modelo.add(Dense(5, input_dim=64, activation="softmax"))
    opt = SGD(learning_rate=0.015)

    modelo.compile(optimizer= opt, loss='categorical_crossentropy', metrics=['accuracy'])


    return modelo, X_hist_train, X_hist_test_entrega, X_hist_test, y_train, y_test

In [ ]:
def entrenamiento(modelo, X_train, X_validacion, X_test, y_train, y_validacion, estadoaleatorio=1, epocas=10):
    parada = EarlyStopping(monitor="val_accuracy", mode="max", patience=80, min_delta=0.0, restore_best_weights=True, verbose=1)
    #parada = EarlyStopping(monitor="val_loss",mode="min",patience=80,min_delta=0.00001,restore_best_weights=True, verbose=1)
    scores, historias = list(), list()
    historia = modelo.fit(X_train, y_train, epochs=epocas, batch_size=32, validation_data=(X_validacion, y_validacion), callbacks=[parada], verbose=1)
    _, acc = modelo.evaluate(X_validacion, y_validacion, verbose=1)
    print('> %.3f' % (acc * 100.0))
    scores.append(acc)
    historias.append(historia)

    return historias, scores, modelo

In [ ]:
def diagnosticos(historias):
	for i in range(len(historias)):
		# graficar loss
		plt.subplot(2, 1, 1)
		plt.title('Cross Entropy Loss')
		plt.plot(historias[i].history['loss'], color='blue', label='train')
		plt.plot(historias[i].history['val_loss'], color='orange', label='test')
		# graficar accuracy
		plt.subplot(2, 1, 2)
		plt.title('Classification Accuracy')
		plt.plot(historias[i].history['accuracy'], color='blue', label='train')
		plt.plot(historias[i].history['val_accuracy'], color='orange', label='test')
	plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes=np.arange(10),
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Purples):
    if not title:
        if normalize:
            title = 'Matriz de confusión normalizada'
        else:
            title = 'Matriz de confusión sin normalizar'
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print('Matriz de confusión normalizada')
    else:
        print('Matriz de confusión sin normalizar')
    print(cm)
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    # We want to show all ticks...
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           # ... and label them with the respective list entries
           xticklabels=np.arange(cm.shape[1]),
           yticklabels=np.arange(cm.shape[0]),
           title=title,
           ylabel='Cifra predicha',
           xlabel='Cifra verdadera')
    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")
    # Loop over data dimensions and create text annotations.
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax
np.set_printoptions(precision=3)

In [ ]:
def matrizdeconfusion(redneuronalentrenada, conjunto, targetsonehot, normalizar=False):
    puntajes = redneuronalentrenada.predict(conjunto)
    prediccion =np.zeros(len(conjunto),dtype=int)
    for x in range(1,len(conjunto)):
        prediccion[x] = np.argmax(puntajes[x])
    plot_confusion_matrix(prediccion, np.argmax(targetsonehot, axis= 1), normalize=normalizar, title= "Matriz de confusión")
    plt.show()

In [ ]:
modelo, X_hist_train, X_hist_test_entrega, X_hist_test, y_train, y_test=miModelo(X_data, y_data, X_test, y_test_original, X_test_entrega)

#El ciclo genera 10 redes neuronales, todas con la misma arquitectura, podrian pero en esta implementacion no tienen valores distintos de entrenamiento.
for i in range(10):
    historias, scores, modelo = entrenamiento(modelo, X_hist_train, X_hist_test, X_hist_test_entrega, y_train, y_test, epocas=1000)
    #GRAFICOS DE COMO FUE ENTRENADO
    plt.rcParams['figure.figsize'] = [8, 8]
    diagnosticos(historias)

    matrizdeconfusion(modelo, X_hist_test, y_test, normalizar=True)
    print(f"Accuracy de validación: {scores[-1] * 100:.2f}%")

    y_pred = modelo.predict(X_hist_test, verbose=0)
    modelo.save(f"modelo_entrenado_promediable_{i}.keras")

In [ ]:
#Promedia los pesos de las redes neuronales calculadas, las mismas pueden diferir en patrones de entrenamiento pero no en su arquitectura.

def weight_averaging(
    rutas_modelos,
    X_validacion,
    y_validacion,
    X_entrega,
    nombres_clases=["gif", "jpg", "png", "qoi", "webp"],
    archivo_modelo="modelo_promediado.keras",
    archivo_predicciones="entrega_weight_averaging.csv"
):

    if len(rutas_modelos) < 2:
        raise ValueError("Se necesitan al menos dos modelos.")

    clases = np.asarray(nombres_clases)

    X_validacion = np.asarray(X_validacion, dtype=np.float32)
    y_validacion = np.asarray(y_validacion)
    X_entrega = np.asarray(X_entrega, dtype=np.float32)

    # 1. CARGAR Y COMPROBAR MODELOS

    modelo_promedio = load_model(
        rutas_modelos[0],
        compile=False
    )

    def firma_arquitectura(modelo):
        return [
            (
                type(capa).__name__,
                {
                    k: v
                    for k, v in capa.get_config().items()
                    if k != "name"
                }
            )
            for capa in modelo.layers
        ]

    firma_base = firma_arquitectura(modelo_promedio)
    pesos_base = modelo_promedio.get_weights()

    if len(modelo_promedio.output_shape) != 2:
        raise ValueError("Se espera una salida de clasificación 2D.")

    if modelo_promedio.output_shape[-1] != len(clases):
        raise ValueError("La cantidad de clases no coincide con el modelo.")

    if X_validacion.ndim != 2 or X_entrega.ndim != 2:
        raise ValueError("Las entradas deben ser matrices 2D.")

    entradas_esperadas = modelo_promedio.input_shape[-1]

    if (X_validacion.shape[1] != entradas_esperadas or
        X_entrega.shape[1] != entradas_esperadas):
        raise ValueError(
            f"El modelo espera {entradas_esperadas} características. "
            f"Validación: {X_validacion.shape[1]}. "
            f"Entrega: {X_entrega.shape[1]}."
        )

    pesos_acumulados = [
        np.zeros_like(p, dtype=np.float32)
        for p in pesos_base
    ]

    for ruta in rutas_modelos:

        modelo = load_model(ruta, compile=False)
        pesos = modelo.get_weights()

        if firma_arquitectura(modelo) != firma_base:
            raise ValueError(f"Arquitectura incompatible: {ruta}")

        if len(pesos) != len(pesos_base):
            raise ValueError(f"Pesos incompatibles: {ruta}")

        for i, (actual, referencia) in enumerate(zip(pesos, pesos_base)):

            if actual.shape != referencia.shape:
                raise ValueError(
                    f"Dimensiones incompatibles en {ruta}, tensor {i}"
                )

            pesos_acumulados[i] += actual.astype(np.float32)

        print("Modelo incorporado:", ruta)

    # 2. PROMEDIAR PESOS Y GUARDAR MODELO

    pesos_promedio = [
        p / len(rutas_modelos)
        for p in pesos_acumulados
    ]

    modelo_promedio.set_weights(pesos_promedio)
    modelo_promedio.save(archivo_modelo)

    print("\nModelo promediado guardado:", archivo_modelo)

    # 3. PREPARAR TARGETS DE VALIDACION

    if y_validacion.ndim == 2:

        # One-Hot Encoding: (500, 5)
        if y_validacion.shape[1] != len(clases):
            raise ValueError("Dimensión OHE incorrecta.")

        y_real = np.argmax(y_validacion, axis=1)

    elif y_validacion.ndim == 1:

        if np.issubdtype(y_validacion.dtype, np.number):
            y_real = y_validacion.astype(int)

            if not np.array_equal(y_real, y_validacion):
                raise ValueError("Los índices de clase deben ser enteros.")

        else:
            mapa = {
                clase.lower(): i
                for i, clase in enumerate(clases)
            }

            y_real = np.array([
                mapa[str(etiqueta).strip().lower()]
                for etiqueta in y_validacion
            ])

    else:
        raise ValueError("Formato de targets no reconocido.")

    if len(y_real) != len(X_validacion):
        raise ValueError(
            "La cantidad de targets no coincide con las muestras."
        )

    if not np.all(np.isin(y_real, np.arange(len(clases)))):
        raise ValueError("Los targets contienen clases desconocidas.")

    # 4. VALIDACION CON EL CONJUNTO EXTERNO

    probabilidades_val = modelo_promedio.predict(
        X_validacion,
        verbose=0
    )

    pred_val = np.argmax(probabilidades_val, axis=1)

    accuracy = accuracy_score(y_real, pred_val)

    print("\n========== VALIDACION EXTERNA ==========")
    print("Muestras evaluadas:", len(y_real))
    print(f"Accuracy: {accuracy * 100:.2f}%")

    print("\nMatriz de confusión:")
    print(confusion_matrix(
        y_real,
        pred_val,
        labels=np.arange(len(clases))
    ))

    print("\nReporte de clasificación:")
    print(classification_report(
        y_real,
        pred_val,
        labels=np.arange(len(clases)),
        target_names=list(clases),
        zero_division=0
    ))

    # 5. PREDICCIONES PARA LA COMPETENCIA

    probabilidades_entrega = modelo_promedio.predict(
        X_entrega,
        verbose=0
    )

    indices_predichos = np.argmax(
        probabilidades_entrega,
        axis=1
    )

    y_pred = clases[indices_predichos]

    pd.DataFrame({
        "y_pred": y_pred
    }).to_csv(
        archivo_predicciones,
        index=False
    )

    print("\n========== ENTREGA ==========")
    print("Muestras clasificadas:", len(y_pred))
    print("Archivo generado:", archivo_predicciones)

    return modelo_promedio, accuracy, y_pred


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

rutas = [
    fr"C:\Users\juane\OneDrive\Desktop\Redes FeedForward\Entregas_clasificacion\modelo_entrenado_promediable_{i}.keras"
    for i in range(7)
]

modelo_promedio, accuracy, y_pred = weight_averaging(
    rutas_modelos=rutas,
    X_validacion=X_hist_test,
    y_validacion=y_test,
    X_entrega=X_hist_test_entrega,
    nombres_clases=["gif", "jpg", "png", "qoi", "webp"]
)
